Licensed to the Apache Software Foundation (ASF) under one
or more contributor license agreements.  See the NOTICE file
distributed with this work for additional information
regarding copyright ownership.  The ASF licenses this file
to you under the Apache License, Version 2.0 (the
"License"); you may not use this file except in compliance
with the License.  You may obtain a copy of the License at

  http://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing,
software distributed under the License is distributed on an
"AS IS" BASIS, WITHOUT WARRANTIES OR CONDITIONS OF ANY
KIND, either express or implied.  See the License for the
specific language governing permissions and limitations
under the License.

In [ ]:
# Execute this cell to install dependencies
%pip install apache-hamilton[openlineage,visualization] sqlalchemy

# OpenLineage example pipeline [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dagworks-inc/hamilton/blob/main/examples/openlineage/notebook.ipynb) [![GitHub badge](https://img.shields.io/badge/github-view_source-2b3137?logo=github)](https://github.com/apache/hamilton/blob/main/examples/openlineage/notebook.ipynb)

This pipeline reads a query joining `orders` and `customers` from a sales database, aggregates daily revenue in Python, and writes `daily_revenue` to a warehouse database. The pipeline does not import OpenLineage and does not need to know about it: the built-in SQL materializers record the datasource they used, and the OpenLineage adapter names the physical tables from that metadata.


In [ ]:
%load_ext hamilton.plugins.jupyter_magic

In [ ]:
%%cell_to_module pipeline --display

"""Revenue reporting with built-in SQL materializers.

- order_lines reads a query joining orders and customers from the sales database.
- daily_revenue aggregates in Python.
- revenue_report writes daily_revenue into the warehouse database.

No custom loader and no hand-written lineage metadata: the SQL materializers record the
datasource they used, and the OpenLineage adapter names the physical tables from it.
"""

import pandas as pd

from hamilton.function_modifiers import load_from, save_to, source, value

REVENUE_QUERY = """
-- paid order lines with the customer's country
WITH paid AS (SELECT * FROM orders WHERE status = 'paid')
SELECT p.order_date, c.country, p.amount
FROM paid p
JOIN customers c ON p.customer_id = c.id
"""


@load_from.sql(query_or_table=value(REVENUE_QUERY), db_connection=source("sales_db"))
def order_lines(df: pd.DataFrame) -> pd.DataFrame:
    return df


def daily_revenue(order_lines: pd.DataFrame) -> pd.DataFrame:
    return order_lines.groupby(["order_date", "country"], as_index=False)["amount"].sum()


@save_to.sql(
    table_name=value("daily_revenue"),
    db_connection=source("warehouse_db"),
    if_exists=value("replace"),
    index=value(False),
    output_name_="saved_revenue",
)
def revenue_report(daily_revenue: pd.DataFrame) -> pd.DataFrame:
    return daily_revenue


# Seed a small sales database and create the OpenLineage client

In [ ]:
import sqlite3
from pathlib import Path

import pandas as pd
from openlineage.client import OpenLineageClient
from openlineage.client.transport.file import FileConfig, FileTransport
from sqlalchemy import create_engine

sales_db = sqlite3.connect("sales.db")
pd.DataFrame(
    {
        "customer_id": [1, 1, 2, 2],
        "order_date": ["2026-09-01", "2026-09-01", "2026-09-01", "2026-09-02"],
        "amount": [10.0, 5.0, 7.5, 3.0],
        "status": ["paid", "paid", "paid", "open"],
    }
).to_sql("orders", sales_db, index=False, if_exists="replace")
pd.DataFrame({"id": [1, 2], "country": ["NL", "DE"]}).to_sql(
    "customers", sales_db, index=False, if_exists="replace"
)
warehouse_db = create_engine(f"sqlite:///{Path('warehouse.db').resolve()}")

# without a running OpenLineage server, the FileTransport writes events to a file
Path("pipeline.json").unlink(missing_ok=True)
client = OpenLineageClient(
    transport=FileTransport(FileConfig(log_file_path="pipeline.json", append=True))
)
# with a running server, e.g. marquez: client = OpenLineageClient(url="http://localhost:5000")

# Run the dataflow with the OpenLineage adapter

In [ ]:
from hamilton import driver
from hamilton.plugins import h_openlineage

adapter = h_openlineage.OpenLineageAdapter(
    client, "demo_namespace", "revenue_job", sql_dataset_identity="datasource"
)
dr = driver.Builder().with_modules(pipeline).with_adapters(adapter).build()
result = dr.execute(["saved_revenue"], inputs={"sales_db": sales_db, "warehouse_db": warehouse_db})
sales_db.close()
result["saved_revenue"]["sql_metadata"]

In [ ]:
# The datasets are named after the datasource (the SQLite files), not the job namespace
import json

for line in Path("pipeline.json").read_text().splitlines():
    event = json.loads(line)
    for kind in ("inputs", "outputs"):
        for dataset in event.get(kind) or []:
            print(f"{event['eventType']} {kind[:-1]}: {dataset['namespace']}  {dataset['name']}")